In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL = "https://www.benty-fields.com/daily_arXiv_results?date=2026-01-09"

def fetch_top10_arxiv_ids():
    resp = requests.get(URL, timeout=10)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    print(soup.prettify)
    papers = []

    # 示例：假设每篇论文在一个 card / list item 里
    for item in soup.select(".paper-item")[:10]:
        # 找 arXiv 链接
        link = item.find("a", href=True)
        if not link:
            continue

        href = link["href"]
        if "arxiv.org" not in href:
            continue

        # 从 URL 中解析 arXiv ID
        # e.g. https://arxiv.org/abs/2401.12345
        arxiv_id = href.rstrip("/").split("/")[-1]
        papers.append(arxiv_id)

    return papers


if __name__ == "__main__":
    ids = fetch_top10_arxiv_ids()
    for i, pid in enumerate(ids, 1):
        print(i, pid)

<bound method Tag.prettify of <!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="yes" name="apple-mobile-web-app-capable"/>
<meta content="" name="author"/>
<title>benty-fields - Login</title>
<meta content="Provide your login credentials." name="description"/>
<link href="/static/favicon/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<link href="/static/favicon/apple-icon-152x152.png" rel="apple-touch-icon" type="image/png"/>
<link href="/static/include/main_combined.min.css" rel="stylesheet"/>
<!-- This is needed for the Patreon symbol in the benty-footer -->
<script src="https://kit.fontawesome.com/a076d05399.js"></script>
<!-- Adding Stripe.js automatically includes advanced fraud detection whenever your customers uses Checkout -->
<script src="https://js.stripe.com/v3/"></script>
<script type="text/javascript">
          

In [ ]:
import requests
from bs4 import BeautifulSoup

session = requests.Session()
login_url = "https://www.benty-fields.com/login"
resp = session.get(login_url)

soup = BeautifulSoup(resp.text, "html.parser")
csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

payload = {
    "email": "",
    "password": "",
    "csrf_token": csrf_token
}

resp = session.post(
    login_url,
    data=payload
)

#check login successfully
if "Logout" not in resp.text:
    raise RuntimeError("Login failed")

url = "https://www.benty-fields.com/daily_arXiv_results?date=2026-01-09"
html = session.get(url).text


In [16]:
soup = BeautifulSoup(html, "html.parser")
papers = soup.find_all("div", class_="paper")[:10]
papers

[<div class="paper" db-source="arxiv" id="paper2601.04344v1" library-source="2601.04344" style="padding-bottom: 0px" value="0">
 <h4 ;="" class="paper_row" style="font-size: 20px">
         
             1. 
             
         
         Nitrogen enhancement of GN-z11 by metal pollution from supermassive stars
     </h4>
 <!--<h4 class="paper_row">Sho Ebihara, Michiko S. Fujii, Takayuki R. Saitoh, Yutaka Hirai, Yuki Isobe, Chris Nagele</h4>-->
 <p class="paper_row">Sho Ebihara, Michiko S. Fujii, Takayuki R. Saitoh, Yutaka Hirai, Yuki Isobe, Chris Nagele</p>
 <div class="paper_row" style="margin-bottom:5px;">
 <div style="padding:0px;margin:0px;margin-bottom:6px;margin-top:0px">
 <h5 class="paper_row">
 <a class="show_abstract" href="javascript:showOrHide_abstract(0);" id="show_abstract0" style="text-decoration:none;">
 <b>
                         
                             Show abstract
                         
                     </b>
 </a> 
 
                 |
             

In [25]:
for paper_div in papers:

    h4 = paper_div.find("h4", class_="paper_row")
    raw_title = h4.get_text(strip=True)

    # "1. Nitrogen enhancement of GN-z11 ..."
    rank_str, title = raw_title.split(".", 1)

    rank = int(rank_str)
    title = title.strip()
    print(title)
    authors = paper_div.find("p", class_="paper_row").get_text(strip=True)
    abs_link = paper_div.find("a", href=lambda x: x and "arxiv.org/abs" in x)

    abs_url = abs_link["href"]
    arxiv_version = abs_link.get_text(strip=True)

    arxiv_id = arxiv_version.split("v")[0]
    print(arxiv_id)
    
    pdf_link = paper_div.find("a", href=lambda x: x and x.endswith(".pdf"))
    pdf_url = pdf_link["href"]
    
    abstract_p = paper_div.find("p", attrs={"name": "abstract_field"})
    abstract = abstract_p.get_text(strip=True)




Nitrogen enhancement of GN-z11 by metal pollution from supermassive stars
2601.04344
The early Universe with JWST and ALMA
2601.04314
Through Thick and Thin: The Cosmic Evolution of Disk Scale Height
2601.04988
Symbolically regressing dark matter halo profiles using weak lensing
2601.05203
Morphologies arising from the gas flow in the innermost kiloparsec of barred galaxy models
2601.04306
Formation of Recycled Pulsars in Common Envelope Binaries
2601.04355
Unveiling the 3D structure of the central molecular zone from stellar kinematics and photometry: The 50 and 20 km/s clouds
2601.05252
Investigating HII Regions in the Disk of NGC 7331 with the Circumgalactic H$α$ Spectrograph
2601.04322
sidmkit: A Reproducible Toolkit for SIDM Phenomenology and Galaxy Rotation-Curve Modeling
2601.04735
First measurement of the Hubble constant from a combined weak lensing and gravitational-wave standard siren analysis
2601.04774


In [29]:
def parse_paper(paper_div):
    h4 = paper_div.find("h4", class_="paper_row")
    raw_title = h4.get_text(strip=True)
    rank_str, title = raw_title.split(".", 1)

    authors = paper_div.find("p", class_="paper_row").get_text(strip=True)

    abs_link = paper_div.find("a", href=lambda x: x and "arxiv.org/abs" in x)
    pdf_link = paper_div.find("a", href=lambda x: x and x.endswith(".pdf"))

    arxiv_version = abs_link.get_text(strip=True)
    arxiv_id = arxiv_version.split("v")[0]

    abstract_p = paper_div.find("p", attrs={"name": "abstract_field"})
    abstract = abstract_p.get_text(strip=True)

    return {
        "rank": int(rank_str),
        "title": title.strip(),
        "authors": authors,
        "abstract": abstract,
        "arxiv_id": arxiv_id,
        "arxiv_version": arxiv_version,
        "pdf_url": pdf_link["href"],
        "abs_url": abs_link["href"],
    }
info = {}
for p in papers:
    paper_info = parse_paper(p)
    info.update(paper_info)
    print(paper_info["rank"], paper_info["arxiv_id"], paper_info["title"])

1 2601.04344 Nitrogen enhancement of GN-z11 by metal pollution from supermassive stars
2 2601.04314 The early Universe with JWST and ALMA
3 2601.04988 Through Thick and Thin: The Cosmic Evolution of Disk Scale Height
4 2601.05203 Symbolically regressing dark matter halo profiles using weak lensing
5 2601.04306 Morphologies arising from the gas flow in the innermost kiloparsec of barred galaxy models
6 2601.04355 Formation of Recycled Pulsars in Common Envelope Binaries
7 2601.05252 Unveiling the 3D structure of the central molecular zone from stellar kinematics and photometry: The 50 and 20 km/s clouds
8 2601.04322 Investigating HII Regions in the Disk of NGC 7331 with the Circumgalactic H$α$ Spectrograph
9 2601.04735 sidmkit: A Reproducible Toolkit for SIDM Phenomenology and Galaxy Rotation-Curve Modeling
10 2601.04774 First measurement of the Hubble constant from a combined weak lensing and gravitational-wave standard siren analysis


In [27]:
from pathlib import Path
import tempfile

TMP_DIR = Path(tempfile.gettempdir()) / "benty_arxiv_pdfs"
TMP_DIR.mkdir(exist_ok=True)

In [33]:

def download_pdf(pdf_url: str, save_path: Path) -> Path:
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    with requests.get(pdf_url, headers=headers, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    return save_path

def download_paper_pdf(paper: dict) -> Path:
    arxiv_id = paper["arxiv_id"]
    pdf_url = paper["pdf_url"]

    pdf_path = TMP_DIR / f"{arxiv_id}.pdf"

    if pdf_path.exists():
        return pdf_path

    return download_pdf(pdf_url, pdf_path)

pdf_paths = []

for p in papers:
    info = parse_paper(p)
    try:
        path = download_paper_pdf(info)
        pdf_paths.append(path)
        print(f"Downloaded: {path.name}")
    except Exception as e:
        print(f"Failed to download {info['arxiv_id']}: {e}")

Failed to download 2601.04344: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
Downloaded: 2601.04314.pdf
Downloaded: 2601.04988.pdf
Downloaded: 2601.05203.pdf
Downloaded: 2601.04306.pdf
Downloaded: 2601.04355.pdf
Downloaded: 2601.05252.pdf
Downloaded: 2601.04322.pdf
Downloaded: 2601.04735.pdf
Downloaded: 2601.04774.pdf


In [32]:
info

{'rank': 10,
 'title': 'First measurement of the Hubble constant from a combined weak lensing and gravitational-wave standard siren analysis',
 'authors': 'Felipe Andrade-Oliveira, David Sanchez-Cid, Danny Laghi, Marcelle Soares-Santos',
 'abstract': "We present a new measurement of the Hubble constant ($H_0$) resulting from the first joint analysis of standard sirens with weak gravitational lensing and galaxy clustering observables comprising three two-point correlation functions (3$\\times$2pt). For the 3$\\times$2pt component of the analysis, we use data from the Dark Energy Survey (DES) Year 3 release. For the standard sirens component, we use data from the Gravitational-Wave Transient Catalog 4.0 released by the LIGO-Virgo-KAGRA (LVK) Collaboration. For GW170817, the only standard siren for which extensive electromagnetic follow-up observations exist, we also use measurements of the host galaxy redshift and inclination angle estimates derived from observations of a superluminal je